# 10 — SEC ↔ Caliper Quantitative Convergence (Idea 3)

Visual validation of the convergence analysis produced by
`scripts/sec_caliper_convergence.py`. The script writes three CSVs under
`results/convergence/sec_caliper/<campaign>/`. This notebook reads them
back, prints summary tables, and draws a per-well 'depth ladder' figure
that places SEC clusters on the left, caliper zones on the right, and
draws connecting lines for matches.

All matching parameters live in `config/pipeline_default.yml` (or in a
user override). They are not hardcoded here.

**Inputs:**
- `results/convergence/sec_caliper/2022_02/cluster_matches.csv`
- `results/convergence/sec_caliper/2022_02/well_summary.csv`
- `results/convergence/sec_caliper/2022_02/unmatched_caliper_zones.csv`
- `results/sec_robustness/2022_02/robustness_clusters.csv` (for the SEC side)
- `data/processed/caliper/priority_wells_cumulative_min_v2_zones.csv` (for the caliper side)

**Output:** one PNG per well under `results/convergence/sec_caliper/<campaign>/figures/`.


In [ ]:
from pathlib import Path
import sys

# Allow this notebook to run whether the kernel was launched from the
# project root or from notebooks/. We make sure src/ is on sys.path.
_HERE = Path.cwd()
_ROOT_CANDIDATES = [_HERE, _HERE.parent, _HERE.parent.parent]
for _r in _ROOT_CANDIDATES:
    if (_r / 'src' / 'karst_analysis').is_dir():
        sys.path.insert(0, str(_r / 'src'))
        ROOT = _r
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from karst_analysis.config import load_config
from karst_analysis.convergence.sec_caliper_match import (
    load_caliper_zones, load_sec_clusters,
)

CAMPAIGN = '2022_02'
OUT = ROOT / 'results' / 'convergence' / 'sec_caliper' / CAMPAIGN
FIG_DIR = OUT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

cfg = load_config()['convergence']['sec_caliper']
cfg

## 1. Load outputs

In [ ]:
cluster_matches = pd.read_csv(OUT / 'cluster_matches.csv')
well_summary    = pd.read_csv(OUT / 'well_summary.csv')
unmatched_zones = pd.read_csv(OUT / 'unmatched_caliper_zones.csv')

# Reload the underlying tables so the figure can show every cluster /
# zone, including ones that were filtered out (e.g. low-agreement
# clusters drawn in grey for context).
all_clusters = load_sec_clusters(
    ROOT / 'results' / 'sec_robustness' / CAMPAIGN / 'robustness_clusters.csv'
)
all_zones = load_caliper_zones(
    ROOT / 'data' / 'processed' / 'caliper'
         / 'priority_wells_cumulative_min_v2_zones.csv'
)

len(cluster_matches), len(well_summary), len(unmatched_zones)

## 2. Per-well summary table

In [ ]:
well_summary.style.format({
    'fraction_converging': '{:.0%}',
    'max_convergence_score': '{:.1f}',
    'mean_convergence_score_converging': '{:.1f}',
})

## 3. Top-converging clusters (ranked)

In [ ]:
cols_show = [
    'well_id', 'depth_median', 'agreement', 'wide_flag',
    'has_caliper_match', 'n_caliper_zones_matched', 'multi_match_flag',
    'max_matched_severity', 'all_matched_severities',
    'best_match_overlap_m', 'best_match_center_distance_m',
    'best_match_thickness_m', 'convergence_score',
]
(cluster_matches
 .sort_values('convergence_score', ascending=False)
 [cols_show]
 .head(15))

## 4. Per-well depth-ladder figure

Two columns per panel:
- **left:** SEC clusters. Bar height = `depth_max − depth_min`. Colour by `agreement`.
- **right:** caliper anomalous zones. Bar height = `thickness_m`. Colour by severity.
- **lines:** connect each analysed cluster to its matched zones.

Y-axis is **BGL-positive** (depth increases downward) following the repo convention.

In [ ]:
SEVERITY_COLOR = {
    'mild':     '#fde68a',  # warm yellow
    'moderate': '#fb923c',  # orange
    'severe':   '#dc2626',  # red
}
AGREEMENT_CMAP = plt.get_cmap('viridis')

def plot_well_ladder(well_id: str, ax: plt.Axes, agreement_min: int) -> None:
    sec  = all_clusters[all_clusters.well_id == well_id]
    cal  = all_zones[all_zones.well_id == well_id]
    matched = cluster_matches[cluster_matches.well_id == well_id]

    # ── SEC column (x ~ 0) ──
    for _, r in sec.iterrows():
        a = r['agreement']
        below_filter = a < agreement_min
        col = '#cbd5e1' if below_filter else AGREEMENT_CMAP(a / 10.0)
        # Bar from depth_min to depth_max (use a tiny min span so single
        # BPs are visible).
        z_lo, z_hi = sorted([r['depth_min'], r['depth_max']])
        span = max(z_hi - z_lo, 0.05)
        ax.barh(y=(z_lo + z_hi) / 2, width=0.35, height=span,
                left=-0.4, color=col, edgecolor='black', linewidth=0.4,
                alpha=0.85 if not below_filter else 0.45)

    # ── Caliper column (x ~ +0.4) ──
    for _, r in cal.iterrows():
        col = SEVERITY_COLOR.get(r['severity'], '#9ca3af')
        z_lo, z_hi = sorted([r['z_bot'], r['z_top']])
        span = max(z_hi - z_lo, 0.05)
        ax.barh(y=(z_lo + z_hi) / 2, width=0.35, height=span,
                left=0.05, color=col, edgecolor='black', linewidth=0.4,
                alpha=0.9)

    # ── Match lines ──
    for _, r in matched.iterrows():
        if not r['has_caliper_match']:
            continue
        # Line from cluster centre to best-match zone centre
        z_zone = (r['best_match_z_top'] + r['best_match_z_bot']) / 2
        sev = r['max_matched_severity']
        line_col = SEVERITY_COLOR.get(sev, '#475569')
        lw = 0.5 + 0.25 * float(r['agreement'])
        ax.plot([-0.05, 0.05], [r['depth_median'], z_zone],
                color=line_col, linewidth=lw, alpha=0.9)

    ax.set_xlim(-0.6, 0.6)
    ax.set_xticks([-0.225, 0.225])
    ax.set_xticklabels(['SEC clusters', 'Caliper zones'])
    ax.invert_yaxis()    # BGL-positive, depth grows downward
    ax.set_ylabel('Depth below ground level (m)')
    ax.set_title(well_id)
    ax.grid(axis='y', linestyle=':', alpha=0.4)

wells_to_plot = sorted(cluster_matches.well_id.unique())
fig, axes = plt.subplots(
    1, len(wells_to_plot),
    figsize=(3 * len(wells_to_plot), 9),
    sharey=False,
)
for ax, w in zip(axes, wells_to_plot):
    plot_well_ladder(w, ax, agreement_min=cfg['sec_agreement_min'])

fig.suptitle(
    f"SEC ↔ Caliper convergence  (campaign {CAMPAIGN}, "
    f"{cfg['matching_rule']} matching, tol = {cfg['tolerance_m']} m)",
    fontsize=12,
)
fig.tight_layout()
fig.savefig(FIG_DIR / 'convergence_ladder_all_wells.png', dpi=160)
fig

## 5. Unmatched caliper zones

These are caliper anomalies (severity ≥ `unmatched_zones_min_severity` from
the config) with **no** matching SEC cluster. They are useful as a parallel
list: candidates for cavities that did not produce a SEC step (e.g. dry
cavities), or potential caliper artefacts.

In [ ]:
unmatched_zones.sort_values(['well_id', 'z_centre'])

## 6. Sanity check — LRS70D piloto

The big severe caliper zone in LRS70D spans 14.65–25.95 m (11.3 m thick).
We expect at least one robust SEC cluster to match part of it. The wide
cluster at 13.27 m sits *above* the zone (gap of ~0.5 m); whether it
matches depends on the chosen tolerance.

In [ ]:
cluster_matches[
    cluster_matches.well_id == 'LRS70D'
].sort_values('depth_median')[cols_show]